# A1.6 · Privilege compromise

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.5 · Tool misuse](https://spbreed.github.io/cyber-commons/lessons/A1.5.html)**.

| | |
|---|---|
| Tools used | Keycloak, SPIFFE/SPIRE |

## What this lesson is

**What it covers.** Have an agent inherit a privileged token and reach something its requester never could.

**Why a security engineer needs it.** The agent acts with more authority than the person who asked it to act, and the log records the service account rather than the human. The control it builds is: delegation that narrows (A2.3), just-in-time grants (A2.4), and default-deny (A3.1).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

An engineer's agent inherits the engineer's standing permissions, because that is the fastest way to make it useful. It now holds production write access at three in the morning, when its principal is asleep and cannot be surprised by anything it does.

> **At CyberTravels.** CyberTravels inherits Alex's standing permissions because that was the fastest way to make it useful. It now holds production write access at three in the morning, when Alex is asleep and cannot be surprised by anything it does. R1.

## 2 · The framework

```
   human principal                      agent
   +-------------------+                +------------------+
   | repo:write        |  inherits ALL  | repo:write       |
   | deploy:prod       | -------------> | deploy:prod      |
   | secrets:read      |   standing     | secrets:read     |
   +-------------------+                +------------------+
     awake 8 hours/day                    awake 24 hours/day
     asked before acting                  acts on retrieved text
```

**OWASP T3 — Privilege Compromise. LLM06 — Excessive Agency.**

Tool misuse is about what a tool can do. Privilege compromise is about **whose
authority it does it with** — the **identity** component rather than the tools
component.

Three patterns produce it, and all three are things teams do for good reasons.

**Inherited human credentials.** The agent runs with the token of the user who
started it. Convenient, and it means the agent holds every permission that user
holds — including the ones irrelevant to the task, and including the ones they
hold because of a role they were given three years ago.

**Shared service accounts.** Every agent authenticates as `agent-svc`. That
account needs the union of everything any agent ever needs, so each agent holds
the maximum of the set.

**Standing scope.** The grant is permanent because renewing it was operationally
awkward. The authority is therefore present at the moment any injection lands.

What makes this distinct from ordinary over-permissioning is the **direction of
the audit trail**. When a human has too much access and misuses it, the log
names them. When an agent does it, the log names the service account — and the
human who caused it is not in the record at all. The compromise is of privilege
*and* of attribution, at the same time.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

A read-only user asks for something, and the agent has more authority than they do.

## 4 · The check, as a skill

Everything turns on which principal the decision was evaluated against. The skill runs the asymmetric probe — a traveller holding `reports:read` asking for something that needs `db:admin` — and then reads one audit row to see whether a human can be recovered from it.

In [ ]:
# skills/threats/authorization-subject-check/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: authorization-subject-check
description: >-
  Establish which principal a tool call is actually authorised against — the
  requesting human, or the agent's own service account — and what the resulting
  audit row can name. Use when a user reaches a scope they do not hold, or when
  every log line shows the same caller.
allowed-tools: Read, Grep, Glob
---

# Authorised as whom?

Privilege compromise in an agentic system is rarely a stolen credential. It is
authorisation evaluated against the **agent's** identity while the request came
from a user who does not hold the scope — and an audit trail that then names
the agent on every row, so the human cannot be recovered at all.

## When to use this

Any time an agent calls something on a user's behalf. Run it before designing
delegation, because the answer decides whether delegation is missing or merely
unenforced.

## Procedure

**1 — Find the authorisation decision point.** The single place where a scope
is compared against a holder. If there are several, list them all; they will
disagree.

**2 — Name the subject at that point.** Is the compared identity the requesting
user, or the process's own account? A service account with a union of every
scope any user might need is the shape to look for.

**3 — Run the asymmetric probe.** Have a user holding a narrow scope request an
action requiring a wider one. If it succeeds, authorisation is on the workload,
not the requester.

**4 — Read one audit row.** Does it name a human? A row naming `agent-svc` is
complete, well-formed and useless: it answers "what ran", never "who caused
it".

**5 — Separate the two fixes.** Authorising on the requester and *recording*
the requester are different changes with different owners; a report that merges
them gets half-implemented.

## Output contract

```json
{
  "decision_points": [{"site": "str", "subject": "requester|workload|both"}],
  "probe": {"user_scopes": ["str"], "required": "str", "succeeded": true},
  "audit_row": {"names_human": false, "fields": ["str"]},
  "fixes": {"authorize_on_requester": true, "record_requester": true}
}
```

## Failure modes

- **Accepting that the user is "in the request".** Present in the payload is
  not the same as compared at the decision point.
- **Testing with an admin.** The probe needs a user who genuinely lacks the
  scope.
- **Reporting the audit gap as a logging bug.** It is the reason the incident
  cannot be scoped later.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/threats/authorization-subject-check/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/threats/authorization-subject-check/scripts/authorization_subject_check.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Show which principal authorisation is actually evaluated against, and what the audit trail can name afterwards.

This is the executable half of the `authorization-subject-check` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

USERS = {"dana":  {"scopes": {"reports:read"}},
         "priya": {"scopes": {"reports:read", "reports:write", "db:admin"}}}

# the agent authenticates as itself, and needs the union of what any user needs
AGENT_SVC = {"name": "agent-svc", "scopes": {"reports:read", "reports:write", "db:admin"}}

AUDIT = []

def call_tool(caller_identity, on_behalf_of, tool, required_scope):
    """Authorization is checked against the CALLER - which is the agent."""
    allowed = required_scope in caller_identity["scopes"]
    AUDIT.append({"actor": caller_identity["name"], "tool": tool,
                  "allowed": allowed})          # note: no human principal
    return allowed

print(f"{'requester':8s}{'their scopes':44s}{'asked for':16s}allowed?")
for user in sorted(USERS):
    ok = call_tool(AGENT_SVC, user, "drop_table", "db:admin")
    print(f"{user:8s}{str(sorted(USERS[user]['scopes'])):44s}{'db:admin':16s}{ok}")

print("\nAUDIT TRAIL")
for a in AUDIT:
    print(f"   actor={a['actor']:10s} tool={a['tool']:12s} allowed={a['allowed']}")

print("\ndana holds reports:read only, and her request reached db:admin.")
print("The authorization decision was made about the agent, not about her.")
print()
print("Now answer 'which user caused the table to be dropped' from that trail.")
print("You cannot: every row says agent-svc. Privilege and attribution failed")
print("in the same step, which is what makes this different from a human with")
print("too much access.")
assert all(a["allowed"] for a in AUDIT)
assert all("dana" not in str(a) for a in AUDIT)

## What you just proved

A user holding only `reports:read` triggers a `db:admin` action, because authorization was evaluated against the shared agent service account rather than the requester — and the audit trail names `agent-svc` on every row, so the human who caused it cannot be recovered from it at all.

## Your turn

Pick one agent and answer two questions: what identity does it authenticate as, and can you name the human behind any single action it took last week. If the second answer is no, you have this risk regardless of how the scopes are set.

---

**Next → [A1.7 · Identity spoofing and impersonation](https://spbreed.github.io/cyber-commons/lessons/A1.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*